# HADIS — LSTM/CNN RF Signal Classifier Training

**High Altitude Drone Intelligence System**  
Author: Prakash Tiwari | Chandigarh Engineering College (IKGPTU)

This notebook trains the hybrid 1D-CNN + Bidirectional LSTM model on the
DroneRF dataset for RF-based drone detection and classification.

- Dataset: DroneRF (AR Drone, Bebop, Phantom, Background)
- Architecture: 3x Conv1D blocks -> BiLSTM -> FC classifier
- Supports resume-from-checkpoint for Colab resilience

---

In [ ]:
# Cell 2 — Install dependencies
!pip install -q torch torchvision scikit-learn matplotlib seaborn

In [ ]:
# Cell 3 — Mount Google Drive and clone HADIS repo
from google.colab import drive
drive.mount('/content/drive')

import os

REPO_DIR = '/content/HADIS'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Tiwari1782/HADIS.git {REPO_DIR}
    print(f'[HADIS] Repository cloned to {REPO_DIR}')
else:
    print(f'[HADIS] Repository already exists at {REPO_DIR}')

In [ ]:
# Cell 4 — Imports and configuration
import sys
import os
import glob
import time

sys.path.append('/content/HADIS')
from config import PATHS, HYPERPARAMS, DRONE_CLASSES, THREAT_LEVELS, NUM_CLASSES

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# Import model
sys.path.append(os.path.join('/content/HADIS', 'hadis-ml'))
from lstm_cnn.model import get_model, HADISRFClassifier

print(f'[HADIS] Config loaded')
print(f'[HADIS] DroneRF path: {PATHS["dronerf"]}')
print(f'[HADIS] LSTM config: seq_len={HYPERPARAMS["lstm_seq_len"]}, epochs={HYPERPARAMS["lstm_epochs"]}, batch={HYPERPARAMS["lstm_batch"]}')

In [ ]:
# Cell 5 — Verify GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    print(f'[HADIS] GPU available: {gpu_name} ({gpu_mem:.1f} GB)')
else:
    print('[HADIS] WARNING: No GPU detected. Training will be slow.')

print(f'[HADIS] Using device: {device}')

In [ ]:
# Cell 6 — DroneRF Dataset class

class DroneRFDataset(Dataset):
    """Dataset for DroneRF signal classification.

    Scans subfolders under PATHS['dronerf'] and maps drone types to classes:
        - AR Drone folder  -> class 0 (consumer_quadcopter)
        - Bebop folder     -> class 0 (consumer_quadcopter)
        - Phantom folder   -> class 0 (consumer_quadcopter)
        - Background RF    -> class 5 (unknown)

    Reads .csv and .npy files, extracts non-overlapping windows of seq_len,
    and normalises each window to zero mean and unit variance.
    """

    # Mapping from subfolder name keywords to class indices
    FOLDER_CLASS_MAP = {
        'ar': 0,         # AR Drone -> consumer_quadcopter
        'bebop': 0,      # Bebop Drone -> consumer_quadcopter
        'bepop': 0,      # Alternate spelling
        'phantom': 0,    # Phantom Drone -> consumer_quadcopter
        'background': 5, # Background RF -> unknown
    }

    def __init__(self, root_dir: str, seq_len: int = 256):
        """Initialise the dataset.

        Args:
            root_dir: Path to DroneRF data directory.
            seq_len: Length of each signal window.
        """
        self.seq_len = seq_len
        self.samples = []  # list of (window_tensor, label)

        if not os.path.exists(root_dir):
            raise FileNotFoundError(f'DroneRF directory not found: {root_dir}')

        print(f'[HADIS] Scanning DroneRF directory: {root_dir}')

        for subfolder in sorted(os.listdir(root_dir)):
            subfolder_path = os.path.join(root_dir, subfolder)
            if not os.path.isdir(subfolder_path):
                continue

            # Determine class label from folder name
            label = None
            folder_lower = subfolder.lower()
            for keyword, cls_idx in self.FOLDER_CLASS_MAP.items():
                if keyword in folder_lower:
                    label = cls_idx
                    break

            if label is None:
                print(f'[HADIS] WARNING: Skipping unrecognised folder: {subfolder}')
                continue

            # Scan for data files
            file_patterns = [
                os.path.join(subfolder_path, '**', '*.csv'),
                os.path.join(subfolder_path, '**', '*.npy'),
            ]
            data_files = []
            for pattern in file_patterns:
                data_files.extend(glob.glob(pattern, recursive=True))

            folder_samples = 0
            for fpath in data_files:
                try:
                    if fpath.endswith('.csv'):
                        df = pd.read_csv(fpath, header=None)
                        signal = df.values.flatten().astype(np.float32)
                    elif fpath.endswith('.npy'):
                        signal = np.load(fpath).flatten().astype(np.float32)
                    else:
                        continue

                    # Extract non-overlapping windows
                    n_windows = len(signal) // seq_len
                    for i in range(n_windows):
                        window = signal[i * seq_len : (i + 1) * seq_len]

                        # Normalise to zero mean, unit variance
                        mean = window.mean()
                        std = window.std()
                        if std > 1e-8:
                            window = (window - mean) / std
                        else:
                            window = window - mean

                        # Shape: (1, seq_len) for Conv1d
                        tensor = torch.tensor(window, dtype=torch.float32).unsqueeze(0)
                        self.samples.append((tensor, label))
                        folder_samples += 1

                except Exception as e:
                    print(f'[HADIS] WARNING: Error reading {fpath}: {e}')
                    continue

            print(f'[HADIS]   {subfolder}: {folder_samples} windows (class {label} = {DRONE_CLASSES[label]})')

        print(f'[HADIS] Total samples: {len(self.samples)}')

    def __len__(self):
        """Return the number of samples."""
        return len(self.samples)

    def __getitem__(self, idx):
        """Return a (signal_window, label) pair."""
        return self.samples[idx]


# Build dataset
seq_len = HYPERPARAMS['lstm_seq_len']
dataset = DroneRFDataset(root_dir=PATHS['dronerf'], seq_len=seq_len)
print(f'[HADIS] Dataset ready: {len(dataset)} samples')

In [ ]:
# Cell 7 — Train/Val/Test split and DataLoaders
total = len(dataset)
train_size = int(0.8 * total)
val_size = int(0.1 * total)
test_size = total - train_size - val_size

generator = torch.Generator().manual_seed(42)
train_ds, val_ds, test_ds = random_split(dataset, [train_size, val_size, test_size], generator=generator)

batch_size = HYPERPARAMS['lstm_batch']
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

print(f'[HADIS] Split: train={train_size}, val={val_size}, test={test_size}')
print(f'[HADIS] Batch size: {batch_size}')

In [ ]:
# Cell 8 — Model, optimizer, and resume-from-checkpoint logic
model = get_model(num_classes=NUM_CLASSES, seq_len=seq_len).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

weights_dir = PATHS['weights_lstm']
os.makedirs(weights_dir, exist_ok=True)

checkpoint_path = os.path.join(weights_dir, 'checkpoint_last.pt')
start_epoch = 0
best_val_acc = 0.0
train_losses, val_losses = [], []
train_accs, val_accs = [], []

if os.path.exists(checkpoint_path):
    try:
        ckpt = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        start_epoch = ckpt['epoch'] + 1
        best_val_acc = ckpt.get('best_val_acc', 0.0)
        train_losses = ckpt.get('train_losses', [])
        val_losses = ckpt.get('val_losses', [])
        train_accs = ckpt.get('train_accs', [])
        val_accs = ckpt.get('val_accs', [])
        print(f'[HADIS] Resumed from epoch {start_epoch} | Best val acc: {best_val_acc:.4f}')
    except Exception as e:
        print(f'[HADIS] WARNING: Failed to load checkpoint: {e}')
        print('[HADIS] Starting fresh training...')
        start_epoch = 0
else:
    print('[HADIS] No checkpoint found. Starting fresh training.')

total_params = sum(p.numel() for p in model.parameters())
print(f'[HADIS] Model parameters: {total_params:,}')

In [ ]:
# Cell 9 — Training loop
num_epochs = HYPERPARAMS['lstm_epochs']

for epoch in range(start_epoch, num_epochs):
    epoch_start = time.time()

    # --- Training phase ---
    model.train()
    running_loss = 0.0
    correct = 0
    total_samples = 0

    for batch_idx, (inputs, labels) in enumerate(train_loader):
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total_samples += inputs.size(0)

    train_loss = running_loss / total_samples
    train_acc = correct / total_samples

    # --- Validation phase ---
    model.eval()
    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            val_correct += predicted.eq(labels).sum().item()
            val_total += inputs.size(0)

    val_loss = val_running_loss / val_total
    val_acc = val_correct / val_total

    scheduler.step(val_loss)

    # Record history
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    elapsed = time.time() - epoch_start
    lr = optimizer.param_groups[0]['lr']
    print(f'[HADIS] Epoch {epoch+1}/{num_epochs} | '
          f'Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | '
          f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | '
          f'LR: {lr:.6f} | Time: {elapsed:.1f}s')

    # --- Save checkpoint every epoch ---
    try:
        ckpt_data = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
            'val_acc': val_acc,
            'best_val_acc': best_val_acc,
            'train_losses': train_losses,
            'val_losses': val_losses,
            'train_accs': train_accs,
            'val_accs': val_accs,
        }
        torch.save(ckpt_data, checkpoint_path)
        print(f'[HADIS] Checkpoint saved: {checkpoint_path}')
    except Exception as e:
        print(f'[HADIS] WARNING: Failed to save checkpoint: {e}')

    # --- Save best model separately ---
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_path = os.path.join(weights_dir, 'hadis_lstm_cnn_best.pt')
        try:
            torch.save(model.state_dict(), best_path)
            print(f'[HADIS] New best model saved: {best_path} (acc={best_val_acc:.4f})')
        except Exception as e:
            print(f'[HADIS] WARNING: Failed to save best model: {e}')

print(f'[HADIS] Training complete. Best validation accuracy: {best_val_acc:.4f}')

In [ ]:
# Cell 10 — Plot loss and accuracy curves
logs_dir = PATHS['logs']
os.makedirs(logs_dir, exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
axes[0].plot(train_losses, label='Train Loss', linewidth=2)
axes[0].plot(val_losses, label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('HADIS LSTM/CNN - Loss Curves')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy plot
axes[1].plot(train_accs, label='Train Accuracy', linewidth=2)
axes[1].plot(val_accs, label='Val Accuracy', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('HADIS LSTM/CNN - Accuracy Curves')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()

try:
    plot_path = os.path.join(logs_dir, 'lstm_cnn_training_curves.png')
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    print(f'[HADIS] Training curves saved to: {plot_path}')
except Exception as e:
    print(f'[HADIS] WARNING: Failed to save plot: {e}')

plt.show()

In [ ]:
# Cell 11 — Test set evaluation: classification report and confusion matrix

# Load best model for evaluation
best_path = os.path.join(weights_dir, 'hadis_lstm_cnn_best.pt')
if os.path.exists(best_path):
    model.load_state_dict(torch.load(best_path, map_location=device))
    print(f'[HADIS] Loaded best model from: {best_path}')
else:
    print('[HADIS] Using last epoch model for evaluation.')

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy() if isinstance(labels, torch.Tensor) else labels)

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Get unique classes present in the test set
unique_classes = sorted(set(all_labels.tolist()) | set(all_preds.tolist()))
target_names = [DRONE_CLASSES[i] for i in unique_classes]

# Classification report
print('[HADIS] ========== Classification Report ==========')
report = classification_report(all_labels, all_preds, labels=unique_classes,
                                target_names=target_names, digits=4)
print(report)

# Confusion matrix heatmap
cm = confusion_matrix(all_labels, all_preds, labels=unique_classes)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names,
            yticklabels=target_names, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('HADIS LSTM/CNN - Confusion Matrix')
plt.tight_layout()

try:
    cm_path = os.path.join(logs_dir, 'lstm_cnn_confusion_matrix.png')
    plt.savefig(cm_path, dpi=150, bbox_inches='tight')
    print(f'[HADIS] Confusion matrix saved to: {cm_path}')
except Exception as e:
    print(f'[HADIS] WARNING: Failed to save confusion matrix: {e}')

plt.show()

# Save classification report to text file
try:
    report_path = os.path.join(logs_dir, 'lstm_cnn_classification_report.txt')
    with open(report_path, 'w') as f:
        f.write('HADIS LSTM/CNN Classification Report\n')
        f.write('=' * 50 + '\n')
        f.write(report)
    print(f'[HADIS] Classification report saved to: {report_path}')
except Exception as e:
    print(f'[HADIS] WARNING: Failed to save report: {e}')